# Phys/APP-Initiated Demographic And CCI Interaction Models

This notebook evaluates whether the association between prior Physician/APP message exposure and patient reply differs by demographic and comorbidity subgroups among Physician/APP-initiated threads.

Scope:
- Load the unified analysis dataset created upstream.
- Verify the manuscript analytic restriction applied upstream and use the same categorical coding as the main analysis notebook.
- Restrict to Physician/APP-initiated thread anchors.
- Fit encounter-adjusted logistic regression models with subgroup interactions for sex, race, age group, and Charlson comorbidity group.
- Export odds-ratio tables and predicted-probability figures.

This notebook does not rebuild threads, message exposure variables, encounter covariates, or the main provider/system analysis models.

Privacy note: row-level EHR-derived datasets are restricted and should not be publicly shared. Share code and aggregate outputs only.


## 1. Setup

In [ ]:
from pathlib import Path
from datetime import datetime
import json
import os
import platform
import sys
import importlib.metadata as importlib_metadata

import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
import matplotlib.pyplot as plt
from scipy.special import expit
from IPython.display import display

pd.options.display.max_columns = 160
pd.options.display.float_format = "{:.3f}".format


def find_repo_root() -> Path:
    cwd = Path.cwd().resolve()
    for candidate in (cwd, *cwd.parents):
        if (candidate / "notebooks").is_dir() and (candidate / "README.md").is_file():
            return candidate
    raise RuntimeError("Run this notebook from within the cloned repository.")


REPO_ROOT = find_repo_root()
OUTPUT_ROOT = Path(os.environ.get("JMIR_REPRO_OUTPUT_DIR", REPO_ROOT / "outputs")).expanduser().resolve()
WORKING_OUTPUT_DIR = Path(
    os.environ.get("JMIR_WORKING_DATA_DIR", OUTPUT_ROOT / "working_data")
).expanduser().resolve()
RUNS_DIR = OUTPUT_ROOT / "runs"
RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S")
RUN_DIR = RUNS_DIR / f"physapp_demographic_cci_interactions_{RUN_ID}"
TABLE_DIR = RUN_DIR / "tables"
FIGURE_DIR = RUN_DIR / "figures"
WORKING_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
TABLE_DIR.mkdir(parents=True, exist_ok=False)
FIGURE_DIR.mkdir(parents=True, exist_ok=False)

UNIFIED_ANALYSIS_PATH = WORKING_OUTPUT_DIR / "analysis_df_all_exposures_with_encounter_covariates.parquet"
if not UNIFIED_ANALYSIS_PATH.exists():
    raise FileNotFoundError(
        f"Missing unified analysis dataset: {UNIFIED_ANALYSIS_PATH}\n"
        "Run 05_build_exposure_analysis_dataset_clean.ipynb first."
    )

OUTCOME = "patient_replied_7d"
PERIODS = ("30d", "90d")
AGE_CONT = "Age_at_Message"
AGE_GROUP = "Age65"
RACE_PREFIX = 'C(Race, Treatment(reference="White"))'
BASE_CONTROLS = ["Ethnicity", "C(Charlson_group)"]
ENCOUNTER_CONTROL = "log_encounters_past_7d"

ANALYTIC_START = pd.Timestamp("2021-03-31")
ANALYTIC_END = pd.Timestamp("2023-11-30 23:59:59.999999999")

print("REPO_ROOT:", REPO_ROOT)
print("OUTPUT_ROOT:", OUTPUT_ROOT)
print("WORKING_OUTPUT_DIR:", WORKING_OUTPUT_DIR)
print("RUN_DIR:", RUN_DIR)
print("UNIFIED_ANALYSIS_PATH:", UNIFIED_ANALYSIS_PATH)

In [ ]:
def package_version(package_name: str) -> str:
    try:
        return importlib_metadata.version(package_name)
    except importlib_metadata.PackageNotFoundError:
        return "not_installed"


def require_columns(df: pd.DataFrame, required: list[str], label: str) -> None:
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(f"{label} is missing required columns: {missing}")


def format_p_jama(p):
    if p is None or pd.isna(p):
        return ""
    p = float(p)
    if p < 0.001:
        return "<.001"
    if p < 0.01:
        return f"{p:.3f}".lstrip("0")
    return f"{p:.2f}".lstrip("0")


def format_or_ci(or_value, lower, upper, decimals=2):
    if any(pd.isna(x) for x in [or_value, lower, upper]):
        return ""
    return f"{or_value:.{decimals}f} ({lower:.{decimals}f}-{upper:.{decimals}f})"


def mode_or_first(series: pd.Series):
    series = series.dropna()
    if series.empty:
        return np.nan
    modes = series.mode()
    return modes.iloc[0] if not modes.empty else series.iloc[0]


def collapse_race(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df["Race"] = np.select(
        [df["Race"].eq("White"), df["Race"].eq("Black/African American"), df["Race"].eq("Asian")],
        ["White", "Black/African American", "Asian"],
        default="Other/Multiple/Unknown",
    )
    df["Race"] = pd.Categorical(
        df["Race"],
        categories=["White", "Black/African American", "Asian", "Other/Multiple/Unknown"],
        ordered=False,
    )
    return df


def ensure_log_encounter(df: pd.DataFrame, encounter_term=ENCOUNTER_CONTROL) -> pd.DataFrame:
    df = df.copy()
    if encounter_term not in df.columns:
        raw_col = encounter_term.replace("log_", "")
        if raw_col in df.columns:
            df[encounter_term] = np.log1p(pd.to_numeric(df[raw_col], errors="coerce"))
        else:
            raise ValueError(f"Missing encounter covariate: {encounter_term}")
    return df

## 2. Load Unified Dataset And Prepare Focused Subset

In [ ]:
focused_raw = pd.read_parquet(UNIFIED_ANALYSIS_PATH)

required_cols = [
    "DMRN", "thread_first_time", "thread_initiator_group",
    OUTCOME, AGE_CONT, AGE_GROUP, "Gender", "Race", "Ethnicity", "Charlson_group",
    ENCOUNTER_CONTROL,
]
require_columns(focused_raw, [c for c in required_cols if c != ENCOUNTER_CONTROL], "focused_raw")

focused = focused_raw.copy()
focused["DMRN"] = focused["DMRN"].astype("string").str.strip().astype(object)
focused["thread_first_time"] = pd.to_datetime(focused["thread_first_time"], errors="coerce")
focused["month_year"] = focused["thread_first_time"].dt.to_period("M").astype(str)

focused["Gender"] = pd.Categorical(focused["Gender"].astype("string").str.strip(), categories=["Female", "Male"], ordered=False)
focused[AGE_GROUP] = pd.Categorical(focused[AGE_GROUP], categories=["<65", "\u226565"], ordered=False)
focused["Ethnicity"] = pd.Categorical(
    focused["Ethnicity"],
    categories=["Not Hispanic/Latino", "Hispanic/Latino", "Unknown/Other"],
    ordered=False,
)
focused["Charlson_group"] = focused["Charlson_group"].astype("category")
focused = collapse_race(focused)
focused = ensure_log_encounter(focused)

start_cutoff = ANALYTIC_START
end_cutoff = ANALYTIC_END

if focused[AGE_GROUP].isna().any():
    raise ValueError("Unified analysis dataset contains missing Age65 values. Rerun notebook 05.")
if not focused["thread_first_time"].between(start_cutoff, end_cutoff, inclusive="both").all():
    raise ValueError("Unified analysis dataset contains rows outside the manuscript analytic period. Rerun notebook 05.")

analysis_physapp = focused.loc[focused["thread_initiator_group"].eq("PhysAPP")].copy()

load_summary = pd.DataFrame({
    "dataset": ["focused_all", "analysis_physapp"],
    "rows": [len(focused), len(analysis_physapp)],
    "patients": [focused["DMRN"].nunique(), analysis_physapp["DMRN"].nunique()],
    "min_thread_time": [focused["thread_first_time"].min(), analysis_physapp["thread_first_time"].min()],
    "max_thread_time": [focused["thread_first_time"].max(), analysis_physapp["thread_first_time"].max()],
})
display(load_summary)

display(analysis_physapp["thread_initiator_group"].value_counts(dropna=False).rename_axis("thread_initiator_group").reset_index(name="threads"))

## 3. Descriptive Summaries

In [ ]:
def n_pct(series):
    denom = int(series.notna().sum())
    count = int(series.fillna(0).astype(int).sum())
    pct = (count / denom * 100) if denom else np.nan
    return f"{count:,} ({pct:.1f}%)", denom


def median_iqr(series, decimals=2):
    s = pd.to_numeric(series, errors="coerce").dropna()
    if s.empty:
        return ""
    q1, med, q3 = np.percentile(s, [25, 50, 75])
    return f"{med:.{decimals}f} ({q1:.{decimals}f}-{q3:.{decimals}f})"


summary_rows = []
for col in ["patient_replied_7d", "patient_replied_14d", "patient_replied_ever"]:
    if col in analysis_physapp.columns:
        value, denom = n_pct(analysis_physapp[col])
        summary_rows.append({"variable": col, "type": "outcome", "summary": value, "N_nonmissing": denom})

for col in [AGE_CONT, ENCOUNTER_CONTROL]:
    summary_rows.append({
        "variable": col,
        "type": "continuous",
        "summary": median_iqr(analysis_physapp[col]),
        "N_nonmissing": int(pd.to_numeric(analysis_physapp[col], errors="coerce").notna().sum()),
    })

for period in PERIODS:
    for col in [
        f"physapp_msgs_past_{period}",
        f"other_human_msgs_past_{period}",
        f"system_msgs_past_{period}",
        f"patient_initiated_threads_past_{period}",
    ]:
        if col in analysis_physapp.columns:
            summary_rows.append({
                "variable": col,
                "type": "count",
                "summary": median_iqr(analysis_physapp[col]),
                "N_nonmissing": int(pd.to_numeric(analysis_physapp[col], errors="coerce").notna().sum()),
            })

physapp_summary = pd.DataFrame(summary_rows)
physapp_summary_path = TABLE_DIR / "table_physapp_interaction_sample_summary.csv"
physapp_summary.to_csv(physapp_summary_path, index=False)
display(physapp_summary)

## 4. Fit Demographic And CCI Interaction Models

Each model includes:
- Phys/APP message exposure.
- Other human message exposure.
- System message exposure, if available.
- Patient-initiated thread history.
- Phys/APP x other-human interaction.
- 7-day encounter control.
- Demographic/comorbidity covariates and month-year fixed effects.

Subgroup models add one interaction at a time: Phys/APP exposure by sex, race, age group, or Charlson comorbidity group.


In [ ]:
def fit_clustered_logit(formula: str, data: pd.DataFrame, cluster_col="DMRN"):
    return smf.logit(formula, data=data).fit(
        cov_type="cluster",
        cov_kwds={"groups": data[cluster_col]},
        disp=False,
    )


def joint_wald_p(model, term_substring):
    terms = [p for p in model.params.index if term_substring in p]
    if not terms:
        return None
    restriction = np.zeros((len(terms), len(model.params)))
    idx = list(model.params.index)
    for row, term in enumerate(terms):
        restriction[row, idx.index(term)] = 1
    return float(model.wald_test(restriction).pvalue)


def resolve_exposure_columns(df: pd.DataFrame, period: str) -> dict:
    x_phys = f"log_physapp_msgs_past_{period}"
    x_other_human = f"log_other_human_msgs_past_{period}"
    x_other_legacy = f"log_other_msgs_past_{period}"
    x_other = x_other_human if x_other_human in df.columns else x_other_legacy
    x_sys = f"log_system_msgs_past_{period}"
    x_pinit = f"log_patient_initiated_threads_past_{period}"
    return {
        "x_phys": x_phys,
        "x_other": x_other,
        "x_sys": x_sys if x_sys in df.columns else None,
        "x_pinit": x_pinit,
        "x_phys_other_int": f"{x_phys}:{x_other}",
    }


def build_terms(exposures: dict, subgroup=None):
    exposure_terms = [
        exposures["x_phys"],
        exposures["x_other"],
        exposures["x_pinit"],
        exposures["x_phys_other_int"],
        ENCOUNTER_CONTROL,
    ]
    if exposures["x_sys"] is not None:
        exposure_terms.insert(2, exposures["x_sys"])

    base_terms = [AGE_CONT, "C(Gender)", RACE_PREFIX] + BASE_CONTROLS + ["C(month_year)"]
    if subgroup == "Sex":
        return exposure_terms + [f"{exposures['x_phys']}:C(Gender)"] + base_terms
    if subgroup == "Race":
        return exposure_terms + [f"{exposures['x_phys']}:{RACE_PREFIX}"] + base_terms
    if subgroup == "Age65":
        return exposure_terms + [f"{exposures['x_phys']}:C({AGE_GROUP})"] + [AGE_CONT, f"C({AGE_GROUP})", "C(Gender)", RACE_PREFIX] + BASE_CONTROLS + ["C(month_year)"]
    if subgroup == "CCI":
        return exposure_terms + [f"{exposures['x_phys']}:C(Charlson_group)"] + base_terms
    return exposure_terms + base_terms


def required_raw_columns(exposures: dict, subgroup=None):
    cols = [
        OUTCOME, "DMRN", "month_year", AGE_CONT, "Gender", "Race", "Ethnicity",
        "Charlson_group", ENCOUNTER_CONTROL, exposures["x_phys"], exposures["x_other"],
        exposures["x_pinit"],
    ]
    if exposures["x_sys"] is not None:
        cols.append(exposures["x_sys"])
    if subgroup == "Age65":
        cols.append(AGE_GROUP)
    return cols


MODEL_HET = {}
model_rows = []

for period in PERIODS:
    exposures = resolve_exposure_columns(analysis_physapp, period)
    model_specs = ["Main", "Sex", "Race", "Age65", "CCI"]

    for model_type in model_specs:
        subgroup = None if model_type == "Main" else model_type
        formula = f"{OUTCOME} ~ " + " + ".join(build_terms(exposures, subgroup=subgroup))
        needed = [c for c in required_raw_columns(exposures, subgroup=subgroup) if c in analysis_physapp.columns]
        df_fit = analysis_physapp.dropna(subset=list(set(needed))).copy()
        df_fit[OUTCOME] = df_fit[OUTCOME].astype(float).fillna(0).astype(int)

        model = fit_clustered_logit(formula, df_fit)
        key = f"PHYSAPP_HET_{model_type}_{period}"

        if model_type == "Sex":
            interaction_p = joint_wald_p(model, f"{exposures['x_phys']}:C(Gender)")
        elif model_type == "Race":
            interaction_p = joint_wald_p(model, f"{exposures['x_phys']}:{RACE_PREFIX}")
        elif model_type == "Age65":
            interaction_p = joint_wald_p(model, f"{exposures['x_phys']}:C({AGE_GROUP})")
        elif model_type == "CCI":
            interaction_p = joint_wald_p(model, f"{exposures['x_phys']}:C(Charlson_group)")
        else:
            interaction_p = None

        phys_other_p = joint_wald_p(model, exposures["x_phys_other_int"])

        MODEL_HET[key] = {
            "key": key,
            "model": model,
            "df_fit": df_fit,
            "period": period,
            "model_type": model_type,
            "outcome": OUTCOME,
            "N": int(df_fit.shape[0]),
            "interaction_p": interaction_p,
            "phys_other_p": phys_other_p,
            **exposures,
            "x_encounter": ENCOUNTER_CONTROL,
        }
        model_rows.append({
            "period": period,
            "model_type": model_type,
            "N": int(df_fit.shape[0]),
            "subgroup_interaction_p": interaction_p,
            "physapp_x_other_p": phys_other_p,
            "AIC": float(model.aic),
            "BIC": float(model.bic),
        })

model_summary = pd.DataFrame(model_rows)
model_summary_path = TABLE_DIR / "table_physapp_demographic_cci_model_summary.csv"
model_summary.to_csv(model_summary_path, index=False)
display(model_summary)

## 5. Odds-Ratio Tables

In [ ]:
def or_ci_p(model, term):
    if term not in model.params.index:
        return np.nan, np.nan, np.nan, np.nan
    beta = float(model.params[term])
    ci = model.conf_int().loc[term].astype(float)
    p_value = float(model.pvalues[term])
    return float(np.exp(beta)), float(np.exp(ci.iloc[0])), float(np.exp(ci.iloc[1])), p_value


def pretty_term(term: str, obj: dict) -> str:
    period = obj["period"]
    mapping = {
        obj["x_phys"]: f"Physician/APP messages (log1p, past {period})",
        obj["x_other"]: f"Other human messages (log1p, past {period})",
        obj["x_pinit"]: f"Patient-initiated threads (log1p, past {period})",
        obj["x_phys_other_int"]: f"Physician/APP x other human (log1p x log1p, past {period})",
        obj["x_encounter"]: "Encounters (log1p, past 7 days)",
        AGE_CONT: "Age at message (years)",
        "C(Gender)[T.Male]": "Male vs Female",
        "Ethnicity[T.Hispanic/Latino]": "Hispanic/Latino vs non-Hispanic",
        "Ethnicity[T.Unknown/Other]": "Unknown/Other vs non-Hispanic",
        f"C({AGE_GROUP})[T.\u226565]": "Age >=65 vs <65",
        "C(Charlson_group)[T.1-2 (Mild)]": "CCI: 1-2 (mild) vs 0",
        "C(Charlson_group)[T.3-4 (Moderate)]": "CCI: 3-4 (moderate) vs 0",
        "C(Charlson_group)[T.5+ (Severe)]": "CCI: 5+ (severe) vs 0",
    }
    if obj["x_sys"] is not None:
        mapping[obj["x_sys"]] = f"System messages (log1p, past {period})"
    if term in mapping:
        return mapping[term]

    if f"{obj['x_phys']}:C(Gender)" in term:
        return f"Physician/APP messages x sex subgroup, {period}"
    if f"{obj['x_phys']}:{RACE_PREFIX}" in term:
        race = term.split("[T.", 1)[1].rstrip("]")
        return f"Physician/APP messages x {race} race subgroup, {period}"
    if f"{obj['x_phys']}:C({AGE_GROUP})" in term:
        return f"Physician/APP messages x age >=65 subgroup, {period}"
    if f"{obj['x_phys']}:C(Charlson_group)" in term:
        cci = term.split("[T.", 1)[1].rstrip("]")
        return f"Physician/APP messages x CCI {cci}, {period}"
    if term.startswith(RACE_PREFIX) and ":" not in term:
        race = term.split("[T.", 1)[1].rstrip("]")
        return f"{race} vs White"
    return term


or_rows = []
for key, obj in MODEL_HET.items():
    model = obj["model"]
    for term in model.params.index:
        if term == "Intercept" or term.startswith("C(month_year)"):
            continue
        or_value, lower, upper, p_value = or_ci_p(model, term)
        or_rows.append({
            "period": obj["period"],
            "model_type": obj["model_type"],
            "term": pretty_term(term, obj),
            "raw_term": term,
            "odds_ratio_95ci": format_or_ci(or_value, lower, upper),
            "p_value": format_p_jama(p_value),
            "N": obj["N"],
            "subgroup_interaction_p": obj["interaction_p"],
            "physapp_x_other_p": obj["phys_other_p"],
        })

or_table = pd.DataFrame(or_rows)
or_table_path = TABLE_DIR / "table_physapp_demographic_cci_interaction_or.csv"
or_table.to_csv(or_table_path, index=False)
display(or_table)

## 6. Predicted-Probability Figures

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator

# Match the established Figure 2 style: grayscale, journal-safe fonts, boxed panel labels,
# inside legends, and PNG export sized for the journal uploader.
plt.rcParams.update({
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "font.family": "sans-serif",
    "font.sans-serif": ["Arial", "Helvetica", "DejaVu Sans"],
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.labelsize": 10,
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
    "legend.fontsize": 9,
    "legend.title_fontsize": 9,
    "savefig.dpi": 300,
})


def mode_or_first(series):
    s = series.dropna()
    if s.empty:
        return None
    m = s.mode()
    return m.iloc[0] if not m.empty else s.iloc[0]


def safe_level(df, col, level, fallback=None):
    vals = set(df[col].dropna().unique())
    if level in vals:
        return level
    return fallback if fallback is not None else mode_or_first(df[col])


def make_base_profile_linear(df, obj, include_month_year=True):
    """Base profile for prediction curves, matching the Figure 2 prediction approach."""
    profile = {}

    for col in ["Ethnicity", "Charlson_group", "Gender", "Race", AGE_GROUP]:
        if col in df.columns:
            profile[col] = mode_or_first(df[col])

    if AGE_CONT in df.columns:
        profile[AGE_CONT] = float(pd.to_numeric(df[AGE_CONT], errors="coerce").median())

    if include_month_year and "month_year" in df.columns:
        profile["month_year"] = mode_or_first(df["month_year"])

    exposure_cols = [obj["x_other"], obj["x_pinit"], obj["x_encounter"]]
    if obj.get("x_sys") is not None:
        exposure_cols.append(obj["x_sys"])

    for col in exposure_cols:
        if col in df.columns:
            profile[col] = float(pd.to_numeric(df[col], errors="coerce").median())

    return profile


def build_curve_df_linear(profile, x_phys_col, physapp_counts, subgroup_col=None, subgroup_level=None):
    rows = []
    for cnt in physapp_counts:
        row = profile.copy()
        row[x_phys_col] = float(np.log1p(cnt))
        if subgroup_col is not None:
            row[subgroup_col] = subgroup_level
        rows.append(row)
    return pd.DataFrame(rows)


def predicted_curve(result, df_curve):
    return result.predict(df_curve).astype(float).values


def add_panel_box_label(ax, letter, title, x_letter=-0.06, y=1.055, x_title_pad=0.08):
    ax.text(
        x_letter, y, letter,
        transform=ax.transAxes,
        va="bottom", ha="left",
        fontsize=11, fontweight="bold",
        bbox=dict(
            boxstyle="square,pad=0.18",
            facecolor="white",
            edgecolor="black",
            linewidth=0.8,
        ),
        clip_on=False,
    )
    ax.text(
        x_letter + x_title_pad, y, title,
        transform=ax.transAxes,
        va="bottom", ha="left",
        fontsize=11,
        clip_on=False,
    )


def legend_inside(ax, title, loc="lower right", ncol=1, xshift=0.10, yshift=0.12):
    anchor_map = {
        "lower right": (1.0, 0.0),
        "lower left": (0.0, 0.0),
        "upper right": (1.0, 1.0),
        "upper left": (0.0, 1.0),
        "center right": (1.0, 0.5),
        "center left": (0.0, 0.5),
        "center": (0.5, 0.5),
        "lower center": (0.5, 0.0),
        "upper center": (0.5, 1.0),
    }
    bx, by = anchor_map.get(loc, (1.0, 0.0))

    ax.legend(
        frameon=False,
        title=title,
        ncol=ncol,
        loc=loc,
        bbox_to_anchor=(bx + xshift, by + yshift),
        bbox_transform=ax.transAxes,
        borderaxespad=0.6,
        handlelength=2.6,
        handletextpad=0.8,
        labelspacing=0.4,
    )


def plot_curves_with_styles(
    ax, x, curves_dict, style_map, period,
    ylim=(0, 100),
    show_ylabel=True,
    y_label="Predicted probability\nof patient reply 7d (%)",
):
    for label, y in curves_dict.items():
        st = style_map.get(label, {})
        ax.plot(
            x, y * 100,
            linewidth=st.get("lw", 2.0),
            linestyle=st.get("ls", "-"),
            color=st.get("color", "0.35"),
            marker=st.get("marker", None),
            markersize=st.get("markersize", 4.6),
            markevery=st.get("markevery", 4),
            markerfacecolor="white",
            markeredgewidth=0.9,
            label=str(label),
        )

    ax.set_xlabel(f"Physician/APP message volume\n(past {period}; capped at p99)")
    ax.xaxis.set_major_locator(MaxNLocator(integer=True))
    ax.set_ylim(*ylim)
    ax.set_yticks(np.arange(0, 61, 10))
    ax.grid(True, alpha=0.25, linestyle=":")
    ax.set_ylabel(y_label if show_ylabel else "")
    ax.tick_params(axis="y", which="both", labelleft=True)


def needed_cols_for_plot(obj, subgroup_col=None):
    cols = [
        OUTCOME,
        AGE_CONT,
        "Gender", "Race", "Ethnicity", "Charlson_group",
        obj["x_phys"], obj["x_other"], obj["x_pinit"], obj["x_encounter"],
    ]
    if obj.get("x_sys") is not None:
        cols.append(obj["x_sys"])
    if subgroup_col is not None:
        cols.append(subgroup_col)
    if "month_year" in obj["df_fit"].columns:
        cols.append("month_year")
    return [c for c in cols if c in obj["df_fit"].columns]


def plot_cci_panel_predprob(
    ax,
    period,
    obj_cci,
    physapp_counts,
    ylim_prob=(0, 40),
    legend_loc="lower left",
    legend_xshift=0.10,
    legend_yshift=0.12,
):
    dfp = obj_cci["df_fit"].dropna(subset=needed_cols_for_plot(obj_cci, "Charlson_group")).copy()

    candidates = [
        ("0 (None)", "0"),
        ("1-2 (Mild)", None),
        ("3-4 (Moderate)", None),
        ("5+ (Severe)", None),
    ]
    existing = set(dfp["Charlson_group"].dropna().unique())
    cci_levels = []
    for primary, fallback in candidates:
        if primary in existing:
            cci_levels.append(primary)
        elif fallback is not None and fallback in existing:
            cci_levels.append(fallback)
    if len(cci_levels) == 0:
        cci_levels = list(dfp["Charlson_group"].dropna().unique())

    profile = make_base_profile_linear(dfp, obj_cci)

    cci_curves = {}
    for cci in cci_levels:
        df_curve = build_curve_df_linear(
            profile=profile,
            x_phys_col=obj_cci["x_phys"],
            physapp_counts=physapp_counts,
            subgroup_col="Charlson_group",
            subgroup_level=cci,
        )
        cci_curves[cci] = predicted_curve(obj_cci["model"], df_curve)

    style_ref = {"color": "0.0", "ls": "-", "lw": 2.2, "marker": "o"}
    style_main1 = {"color": "0.30", "ls": "--", "lw": 2.2, "marker": "s"}
    style_mild = {"color": "0.55", "ls": (0, (1, 1)), "lw": 2.0, "marker": "^"}
    style_sev = {"color": "0.70", "ls": (0, (3, 1, 1, 1)), "lw": 2.0, "marker": "D"}

    cci_style = {}
    for lvl in cci_levels:
        s = str(lvl)
        if ("0 (None)" in s) or (s.strip() == "0") or ("None" in s and "0" in s):
            cci_style[lvl] = style_ref
        elif ("3-4" in s) or ("Moderate" in s):
            cci_style[lvl] = style_main1
        elif ("1-2" in s) or ("Mild" in s):
            cci_style[lvl] = style_mild
        elif ("5+" in s) or ("Severe" in s):
            cci_style[lvl] = style_sev
        else:
            cci_style[lvl] = style_sev

    plot_curves_with_styles(
        ax=ax,
        x=physapp_counts,
        curves_dict=cci_curves,
        style_map=cci_style,
        period=period,
        ylim=ylim_prob,
        show_ylabel=False,
    )

    handles, labels = ax.get_legend_handles_labels()
    ax.legend(
        handles[::-1], labels[::-1],
        frameon=False,
        title="Charlson group",
        loc=legend_loc,
        bbox_to_anchor=(1.0 + legend_xshift, 0.0 + legend_yshift),
        bbox_transform=ax.transAxes,
        borderaxespad=0.6,
        handlelength=2.6,
        handletextpad=0.8,
        labelspacing=0.4,
    )

    return cci_levels


def make_fig_period_linear_3panels_jama_sex_race_cci(
    period,
    obj_sex,
    obj_race,
    obj_cci,
    pdf_name=None,
    png_name=None,
    show=True,
    ylim_prob=(0, 70),
    xcap_quantile=0.99,
    fallback_cap=35,
    leg_loc_A="lower right",
    leg_loc_B="lower right",
    leg_loc_C="lower right",
    leg_xshift=0.10,
    leg_yshift=0.12,
):
    raw_phys_col = f"physapp_msgs_past_{period}"
    df_for_cap = obj_sex["df_fit"]
    if raw_phys_col in df_for_cap.columns:
        cap = pd.to_numeric(df_for_cap[raw_phys_col], errors="coerce").quantile(xcap_quantile)
        max_physapp = int(np.floor(cap)) if np.isfinite(cap) else fallback_cap
        max_physapp = max(max_physapp, 1)
    else:
        max_physapp = fallback_cap

    physapp_counts = list(range(0, max_physapp + 1))

    dfp_sex = obj_sex["df_fit"].dropna(subset=needed_cols_for_plot(obj_sex, "Gender")).copy()
    dfp_race = obj_race["df_fit"].dropna(subset=needed_cols_for_plot(obj_race, "Race")).copy()

    profile_sex = make_base_profile_linear(dfp_sex, obj_sex)
    profile_race = make_base_profile_linear(dfp_race, obj_race)

    female = safe_level(dfp_sex, "Gender", "Female")
    male = safe_level(dfp_sex, "Gender", "Male", fallback=female)
    sex_levels = [male, female]

    r_ref = safe_level(dfp_race, "Race", "White")
    race_levels_show = ["Asian", "Black/African American", "Other/Multiple/Unknown", "White"]
    race_levels = list(dict.fromkeys([safe_level(dfp_race, "Race", r, fallback=r_ref) for r in race_levels_show]))

    sex_curves = {
        g: predicted_curve(
            obj_sex["model"],
            build_curve_df_linear(profile_sex, obj_sex["x_phys"], physapp_counts, "Gender", g),
        )
        for g in sex_levels
    }
    race_curves = {
        r: predicted_curve(
            obj_race["model"],
            build_curve_df_linear(profile_race, obj_race["x_phys"], physapp_counts, "Race", r),
        )
        for r in race_levels
    }

    style_ref = {"color": "0.0", "ls": "-", "lw": 2.2, "marker": "o"}
    style_main1 = {"color": "0.30", "ls": "--", "lw": 2.2, "marker": "s"}
    style_main2 = {"color": "0.55", "ls": (0, (1, 1)), "lw": 2.0, "marker": "^"}
    style_other = {"color": "0.70", "ls": (0, (3, 1, 1, 1)), "lw": 2.0, "marker": "D"}

    sex_style = {female: style_ref, male: style_main1}

    race_style = {r_ref: style_ref}
    for r in race_levels:
        if r == r_ref:
            continue
        if r == "Asian":
            race_style[r] = style_main2
        elif r == "Black/African American":
            race_style[r] = style_main1
        elif r == "Other/Multiple/Unknown":
            race_style[r] = style_other
        else:
            race_style[r] = style_other

    fig, axes = plt.subplots(1, 3, figsize=(13.2, 4.4), dpi=300)

    plot_curves_with_styles(
        axes[0], physapp_counts, sex_curves,
        style_map=sex_style, period=period, ylim=ylim_prob,
        show_ylabel=True,
    )
    add_panel_box_label(axes[0], "A", "Sex")
    legend_inside(axes[0], title="Sex", loc=leg_loc_A, ncol=1,
                  xshift=leg_xshift, yshift=leg_yshift)

    plot_curves_with_styles(
        axes[1], physapp_counts, race_curves,
        style_map=race_style, period=period, ylim=ylim_prob,
        show_ylabel=False,
    )
    add_panel_box_label(axes[1], "B", "Race")

    handles, labels = axes[1].get_legend_handles_labels()
    labels = ["Other" if lab == "Other/Multiple/Unknown" else lab for lab in labels]
    base_anchor = (1.0, 0.0) if leg_loc_B == "lower right" else (0.0, 0.0)
    axes[1].legend(
        handles, labels,
        frameon=False,
        title="Race",
        ncol=1,
        loc=leg_loc_B,
        bbox_to_anchor=(base_anchor[0] + leg_xshift, base_anchor[1] + leg_yshift),
        bbox_transform=axes[1].transAxes,
        borderaxespad=0.6,
        handlelength=2.6,
        handletextpad=0.8,
        labelspacing=0.4,
    )

    plot_cci_panel_predprob(
        ax=axes[2],
        period=period,
        obj_cci=obj_cci,
        physapp_counts=physapp_counts,
        ylim_prob=ylim_prob,
        legend_loc=leg_loc_C,
        legend_xshift=leg_xshift,
        legend_yshift=leg_yshift,
    )
    add_panel_box_label(axes[2], "C", "Charlson comorbidity index")

    plt.subplots_adjust(bottom=0.14, wspace=0.24, top=0.88)

    if pdf_name is not None:
        fig.savefig(pdf_name, format="pdf", bbox_inches="tight", pad_inches=0.05)

    if png_name is not None:
        fig_w_in = fig.get_size_inches()[0]
        png_dpi = 300
        fig.savefig(
            png_name,
            format="png",
            dpi=png_dpi,
            bbox_inches="tight",
            pad_inches=0.05,
            facecolor="white",
            edgecolor="white",
        )
        print(f"Saved: {png_name} (dpi={png_dpi}, width~{int(fig_w_in * png_dpi)} px)")

    if show:
        plt.show()
    else:
        plt.close(fig)

    return fig, max_physapp


figure_files = []
figure_caps = {}
for period in PERIODS:
    obj_sex = MODEL_HET[f"PHYSAPP_HET_Sex_{period}"]
    obj_race = MODEL_HET[f"PHYSAPP_HET_Race_{period}"]
    obj_cci = MODEL_HET[f"PHYSAPP_HET_CCI_{period}"]

    pdf_path = FIGURE_DIR / f"Figure2_PhysAPP_Demographic_CCI_Interactions_{period}_encounter7d_adjusted.pdf"
    png_path = FIGURE_DIR / f"Figure2_PhysAPP_Demographic_CCI_Interactions_{period}_encounter7d_adjusted.png"

    fig, cap = make_fig_period_linear_3panels_jama_sex_race_cci(
        period=period,
        obj_sex=obj_sex,
        obj_race=obj_race,
        obj_cci=obj_cci,
        pdf_name=pdf_path,
        png_name=png_path,
        show=True,
        xcap_quantile=0.99,
        ylim_prob=(0, 62),
        leg_xshift=0.12,
        leg_yshift=0.10,
    )
    figure_caps[period] = cap
    figure_files.extend([pdf_path, png_path])

print("Saved figures:")
for path in figure_files:
    print(" -", path)
print("Physician/APP p99 caps:", figure_caps)

## 7. Output Index

In [ ]:
qa_summary = {
    "run_id": RUN_ID,
    "run_timestamp": datetime.now().isoformat(timespec="seconds"),
    "python_version": sys.version,
    "platform": platform.platform(),
    "package_versions": {
        "numpy": np.__version__,
        "pandas": pd.__version__,
        "statsmodels": package_version("statsmodels"),
        "matplotlib": package_version("matplotlib"),
        "scipy": package_version("scipy"),
        "pyarrow": package_version("pyarrow"),
    },
    "input_files": {
        "unified_analysis": str(UNIFIED_ANALYSIS_PATH),
    },
    "row_counts": {
        "canonical_rows_loaded": int(len(focused)),
        "physapp_rows": int(len(analysis_physapp)),
        "physapp_patients": int(analysis_physapp["DMRN"].nunique()),
    },
    "output_directories": {
        "tables": str(TABLE_DIR),
        "figures": str(FIGURE_DIR),
    },
    "privacy_restriction": "Row-level EHR-derived datasets are restricted and should not be publicly shared. Share code and aggregate outputs only.",
}

qa_path = RUN_DIR / "qa_physapp_demographic_cci_interactions_summary.json"
with open(qa_path, "w", encoding="utf-8") as f:
    json.dump(qa_summary, f, indent=2, default=str)

saved_files = sorted([p for p in RUN_DIR.rglob("*") if p.is_file()])
print("Saved files:")
for path in saved_files:
    print(" -", path)

---

## 8. No Encounter Control Comparison Specification

The primary models above include the 7-day pre-thread encounter control. This final section fits the same Physician/APP-initiated demographic and CCI interaction specifications **without** the encounter control. It is separated from the primary analysis and writes comparison outputs with `_no_encounter` file suffixes.

This section uses the same filtered `analysis_physapp` dataset, outcome, exposure windows, clustered standard errors, month-year fixed effects, and subgroup-interaction structure as the encounter-adjusted models above. The only specification difference is that `log_encounters_past_7d` is omitted.


In [ ]:
def build_terms_no_encounter(exposures: dict, subgroup=None):
    exposure_terms = [
        exposures["x_phys"],
        exposures["x_other"],
        exposures["x_pinit"],
        exposures["x_phys_other_int"],
    ]
    if exposures["x_sys"] is not None:
        exposure_terms.insert(2, exposures["x_sys"])

    base_terms = [AGE_CONT, "C(Gender)", RACE_PREFIX] + BASE_CONTROLS + ["C(month_year)"]
    if subgroup == "Sex":
        return exposure_terms + [f"{exposures['x_phys']}:C(Gender)"] + base_terms
    if subgroup == "Race":
        return exposure_terms + [f"{exposures['x_phys']}:{RACE_PREFIX}"] + base_terms
    if subgroup == "Age65":
        return exposure_terms + [f"{exposures['x_phys']}:C({AGE_GROUP})"] + [AGE_CONT, f"C({AGE_GROUP})", "C(Gender)", RACE_PREFIX] + BASE_CONTROLS + ["C(month_year)"]
    if subgroup == "CCI":
        return exposure_terms + [f"{exposures['x_phys']}:C(Charlson_group)"] + base_terms
    return exposure_terms + base_terms


def required_raw_columns_no_encounter(exposures: dict, subgroup=None):
    cols = [
        OUTCOME, "DMRN", "month_year", AGE_CONT, "Gender", "Race", "Ethnicity",
        "Charlson_group", exposures["x_phys"], exposures["x_other"], exposures["x_pinit"],
    ]
    if exposures["x_sys"] is not None:
        cols.append(exposures["x_sys"])
    if subgroup == "Age65":
        cols.append(AGE_GROUP)
    return cols


MODEL_HET_NO_ENCOUNTER = {}
model_rows_no_encounter = []

for period in PERIODS:
    exposures = resolve_exposure_columns(analysis_physapp, period)
    model_specs = ["Main", "Sex", "Race", "Age65", "CCI"]

    for model_type in model_specs:
        subgroup = None if model_type == "Main" else model_type
        formula = f"{OUTCOME} ~ " + " + ".join(build_terms_no_encounter(exposures, subgroup=subgroup))
        needed = [c for c in required_raw_columns_no_encounter(exposures, subgroup=subgroup) if c in analysis_physapp.columns]
        df_fit = analysis_physapp.dropna(subset=list(set(needed))).copy()
        df_fit[OUTCOME] = df_fit[OUTCOME].astype(float).fillna(0).astype(int)

        model = fit_clustered_logit(formula, df_fit)
        key = f"PHYSAPP_HET_NOENCOUNTER_{model_type}_{period}"

        if model_type == "Sex":
            interaction_p = joint_wald_p(model, f"{exposures['x_phys']}:C(Gender)")
        elif model_type == "Race":
            interaction_p = joint_wald_p(model, f"{exposures['x_phys']}:{RACE_PREFIX}")
        elif model_type == "Age65":
            interaction_p = joint_wald_p(model, f"{exposures['x_phys']}:C({AGE_GROUP})")
        elif model_type == "CCI":
            interaction_p = joint_wald_p(model, f"{exposures['x_phys']}:C(Charlson_group)")
        else:
            interaction_p = None

        phys_other_p = joint_wald_p(model, exposures["x_phys_other_int"])

        MODEL_HET_NO_ENCOUNTER[key] = {
            "key": key,
            "model": model,
            "df_fit": df_fit,
            "period": period,
            "model_type": model_type,
            "outcome": OUTCOME,
            "N": int(df_fit.shape[0]),
            "interaction_p": interaction_p,
            "phys_other_p": phys_other_p,
            **exposures,
            "x_encounter": None,
            "encounter_control_included": False,
        }
        model_rows_no_encounter.append({
            "period": period,
            "model_type": model_type,
            "N": int(df_fit.shape[0]),
            "encounter_control_included": False,
            "subgroup_interaction_p": interaction_p,
            "physapp_x_other_p": phys_other_p,
            "AIC": float(model.aic),
            "BIC": float(model.bic),
        })

model_summary_no_encounter = pd.DataFrame(model_rows_no_encounter)
model_summary_no_encounter_path = TABLE_DIR / "table_physapp_demographic_cci_model_summary_no_encounter.csv"
model_summary_no_encounter.to_csv(model_summary_no_encounter_path, index=False)
display(model_summary_no_encounter)

In [ ]:
def pretty_term_no_encounter(term: str, obj: dict) -> str:
    period = obj["period"]
    mapping = {
        obj["x_phys"]: f"Physician/APP messages (log1p, past {period})",
        obj["x_other"]: f"Other human messages (log1p, past {period})",
        obj["x_pinit"]: f"Patient-initiated threads (log1p, past {period})",
        obj["x_phys_other_int"]: f"Physician/APP x other human (log1p x log1p, past {period})",
        AGE_CONT: "Age at message (years)",
        "C(Gender)[T.Male]": "Male vs Female",
        "Ethnicity[T.Hispanic/Latino]": "Hispanic/Latino vs non-Hispanic",
        "Ethnicity[T.Unknown/Other]": "Unknown/Other vs non-Hispanic",
        f"C({AGE_GROUP})[T.{chr(0x2265)}65]": "Age >=65 vs <65",
        "C(Charlson_group)[T.1-2 (Mild)]": "CCI: 1-2 (mild) vs 0",
        "C(Charlson_group)[T.3-4 (Moderate)]": "CCI: 3-4 (moderate) vs 0",
        "C(Charlson_group)[T.5+ (Severe)]": "CCI: 5+ (severe) vs 0",
    }
    if obj["x_sys"] is not None:
        mapping[obj["x_sys"]] = f"System messages (log1p, past {period})"
    if term in mapping:
        return mapping[term]

    if f"{obj['x_phys']}:C(Gender)" in term:
        return f"Physician/APP messages x sex subgroup, {period}"
    if f"{obj['x_phys']}:{RACE_PREFIX}" in term:
        race = term.split("[T.", 1)[1].rstrip("]")
        return f"Physician/APP messages x {race} race subgroup, {period}"
    if f"{obj['x_phys']}:C({AGE_GROUP})" in term:
        return f"Physician/APP messages x age >=65 subgroup, {period}"
    if f"{obj['x_phys']}:C(Charlson_group)" in term:
        cci = term.split("[T.", 1)[1].rstrip("]")
        return f"Physician/APP messages x CCI {cci}, {period}"
    if term.startswith(RACE_PREFIX) and ":" not in term:
        race = term.split("[T.", 1)[1].rstrip("]")
        return f"{race} vs White"
    return term


or_rows_no_encounter = []
for key, obj in MODEL_HET_NO_ENCOUNTER.items():
    model = obj["model"]
    for term in model.params.index:
        if term == "Intercept" or term.startswith("C(month_year)"):
            continue
        or_value, lower, upper, p_value = or_ci_p(model, term)
        or_rows_no_encounter.append({
            "period": obj["period"],
            "model_type": obj["model_type"],
            "encounter_control_included": False,
            "term": pretty_term_no_encounter(term, obj),
            "raw_term": term,
            "odds_ratio_95ci": format_or_ci(or_value, lower, upper),
            "p_value": format_p_jama(p_value),
            "N": obj["N"],
            "subgroup_interaction_p": obj["interaction_p"],
            "physapp_x_other_p": obj["phys_other_p"],
        })

or_table_no_encounter = pd.DataFrame(or_rows_no_encounter)
or_table_no_encounter_path = TABLE_DIR / "table_physapp_demographic_cci_interaction_or_no_encounter.csv"
or_table_no_encounter.to_csv(or_table_no_encounter_path, index=False)
display(or_table_no_encounter)

print("Saved no-encounter-control comparison outputs:")
print(" -", model_summary_no_encounter_path)
print(" -", or_table_no_encounter_path)

## Aggregate subgroup-model handoff

Export the sex-, race-, and comorbidity-interaction models for notebook 10. These are the fitted 30-day models used for Figure 2, with modal categorical and median continuous covariates.


In [ ]:
import sys
if str(REPO_ROOT) not in sys.path:sys.path.insert(0,str(REPO_ROOT))
from analysis.model_exports import export_subgroups
SUBGROUP_AGGREGATES=export_subgroups(MODEL_HET,make_base_profile_linear,RUN_DIR)
print(SUBGROUP_AGGREGATES)
